# **IMPLEMENTASI FORWARD DAN BACKWARD CHAINING**

## **Pendahuluan**

Dalam sistem berbasis pengetahuan, proses inferensi merupakan mekanisme utama yang digunakan untuk menghasilkan kesimpulan dari sekumpulan fakta dan aturan. Dua pendekatan utama yang digunakan adalah forward chaining dan backward chaining.

Forward chaining merupakan metode penalaran berbasis data yang dimulai dari fakta awal, kemudian secara iteratif mengeksekusi aturan yang relevan hingga mencapai kesimpulan. Metode ini cocok digunakan pada sistem monitoring atau sistem yang menerima input secara kontinu.

Sebaliknya, backward chaining merupakan metode penalaran berbasis tujuan. Proses dimulai dari hipotesis atau tujuan yang ingin dibuktikan, kemudian sistem akan menelusuri aturan secara mundur untuk memverifikasi apakah fakta yang mendukung tersedia.

Notebook ini menyajikan dua studi kasus sederhana untuk memperlihatkan perbedaan mekanisme kedua metode tersebut secara komputasional dan transparan.

# **A. Forward Chaining**
## **Studi Kasus Diagnosa Penyakit Flu**

### **1. Rule Based Penyakit Flu**

In [6]:
rules_fc = [
    {"id": "R1", "if": ["demam", "batuk"], "then": "infeksi"},
    {"id": "R2", "if": ["infeksi"], "then": "radang tenggorokan"},
    {"id": "R3", "if": ["radang tenggorokan"], "then": "flu"},
    {"id": "R4", "if": ["flu"], "then": "butuh istirahat"},
    {"id": "R5", "if": ["flu"], "then": "minum obat"}
]

print("=== RULE BASE ===")
for r in rules_fc:
    print(f"{r['id']} : IF {r['if']} THEN {r['then']}")

=== RULE BASE ===
R1 : IF ['demam', 'batuk'] THEN infeksi
R2 : IF ['infeksi'] THEN radang tenggorokan
R3 : IF ['radang tenggorokan'] THEN flu
R4 : IF ['flu'] THEN butuh istirahat
R5 : IF ['flu'] THEN minum obat


### **2. Fakta Awal**

In [7]:
facts_fc = set(["demam", "batuk"])

print("=== FAKTA AWAL ===")
for f in facts_fc:
    print("-", f)

=== FAKTA AWAL ===
- batuk
- demam


### **3. Preview Rule Aktif**

In [8]:
print("=== RULE YANG AKTIF ===")
for rule in rules_fc:
    if all(cond in facts_fc for cond in rule["if"]):
        print(f"{rule['id']} dapat dijalankan → {rule['then']}")

=== RULE YANG AKTIF ===
R1 dapat dijalankan → infeksi


### **4. Forward Chaining**

In [9]:
def forward_chaining_verbose(rules, facts):
    facts = set(facts)
    iteration = 1
    
    print("\n=== PROSES FORWARD CHAINING ===")
    
    while True:
        new_fact = False
        print(f"\n--- ITERASI {iteration} ---")
        
        for rule in rules:
            if all(cond in facts for cond in rule["if"]):
                if rule["then"] not in facts:
                    
                    print(f"[FIRE] {rule['id']}")
                    print(f"Premis terpenuhi: {rule['if']}")
                    print(f"Fakta baru: {rule['then']}\n")
                    
                    facts.add(rule["then"])
                    new_fact = True
        
        print("Fakta saat ini:")
        for f in facts:
            print("•", f)
        
        if not new_fact:
            print("\nTidak ada rule tambahan.")
            break
        
        iteration += 1
    
    return facts

### **5. Eksekusi**

In [5]:
final_fc = forward_chaining_verbose(rules_fc, facts_fc)

print("\n=== HASIL AKHIR ===")
for f in final_fc:
    print("-", f)


=== PROSES FORWARD CHAINING ===

--- ITERASI 1 ---
[FIRE] R1
Premis terpenuhi: ['demam', 'batuk']
Fakta baru: infeksi

[FIRE] R2
Premis terpenuhi: ['infeksi']
Fakta baru: radang tenggorokan

[FIRE] R3
Premis terpenuhi: ['radang tenggorokan']
Fakta baru: flu

[FIRE] R4
Premis terpenuhi: ['flu']
Fakta baru: butuh istirahat

[FIRE] R5
Premis terpenuhi: ['flu']
Fakta baru: minum obat

Fakta saat ini:
• minum obat
• butuh istirahat
• flu
• infeksi
• radang tenggorokan
• batuk
• demam

--- ITERASI 2 ---
Fakta saat ini:
• minum obat
• butuh istirahat
• flu
• infeksi
• radang tenggorokan
• batuk
• demam

Tidak ada rule tambahan.

=== HASIL AKHIR ===
- minum obat
- butuh istirahat
- flu
- infeksi
- radang tenggorokan
- batuk
- demam


Terlihat bahwa:

- reasoning berjalan berlapis
- setiap fakta baru membuka rule berikutnya
- sistem melakukan chain propagation

# **B. Backward Chaining**
## **Kelayakan Penerima Beasiswa Djarum Beasiswa Plus**

### **1. Rule Base Beasiswa**

In [10]:
rules_bc = [
    {"id": "R1", "if": ["IPK >= 3.5", "aktif organisasi"], "then": "berprestasi"},
    {"id": "R2", "if": ["berprestasi"], "then": "layak beasiswa"},
    {"id": "R3", "if": ["layak beasiswa"], "then": "diterima"}
]

print("=== RULE BASE ===")
for r in rules_bc:
    print(f"{r['id']} : IF {r['if']} THEN {r['then']}")

=== RULE BASE ===
R1 : IF ['IPK >= 3.5', 'aktif organisasi'] THEN berprestasi
R2 : IF ['berprestasi'] THEN layak beasiswa
R3 : IF ['layak beasiswa'] THEN diterima


### **2. Fakta**

In [11]:
facts_bc = set(["IPK >= 3.5", "aktif organisasi"])

print("=== FAKTA ===")
for f in facts_bc:
    print("-", f)

=== FAKTA ===
- aktif organisasi
- IPK >= 3.5


### **3. Backward Chaining**

In [12]:
def backward_chaining(goal, rules, facts, depth=0):
    indent = "  " * depth
    print(f"{indent}Cek goal: {goal}")
    
    if goal in facts:
        print(f"{indent}✔ Ditemukan di fakta\n")
        return True
    
    for rule in rules:
        if rule["then"] == goal:
            print(f"{indent}Gunakan {rule['id']}")
            
            all_proven = True
            for cond in rule["if"]:
                if not backward_chaining(cond, rules, facts, depth+1):
                    all_proven = False
            
            if all_proven:
                print(f"{indent}✔ Goal {goal} terbukti\n")
                return True
    
    print(f"{indent}✘ Goal {goal} gagal\n")
    return False

### **4. Eksekusi Backward Chaining**

In [13]:
goal = "diterima"

print("=== PROSES BACKWARD CHAINING ===\n")
result = backward_chaining(goal, rules_bc, facts_bc)

print("=== HASIL ===")
if result:
    print("Mahasiswa diterima beasiswa")
else:
    print("Mahasiswa tidak diterima")

=== PROSES BACKWARD CHAINING ===

Cek goal: diterima
Gunakan R3
  Cek goal: layak beasiswa
  Gunakan R2
    Cek goal: berprestasi
    Gunakan R1
      Cek goal: IPK >= 3.5
      ✔ Ditemukan di fakta

      Cek goal: aktif organisasi
      ✔ Ditemukan di fakta

    ✔ Goal berprestasi terbukti

  ✔ Goal layak beasiswa terbukti

✔ Goal diterima terbukti

=== HASIL ===
Mahasiswa diterima beasiswa


Ini menunjukkan:

- recursive goal checking
- depth-first reasoning
- verifikasi berbasis hipotesis

| Aspek   | Forward           | Backward        |
| ------- | ----------------- | --------------- |
| Pola    | Propagasi         | Rekursif        |
| Depth   | Linear chain      | Tree search     |
| Trigger | Fakta             | Goal            |
| Output  | Semua kemungkinan | Target spesifik |
